<a href="https://colab.research.google.com/github/TomasVargas1911/Entregable-BIT-Tomas-Vargas/blob/main/Actividad_2_Limpieza_de_datos_con_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 2: Limpieza y análisis de datos con Pandas + NumPy

**Dataset:** Animal Data - Dirty Data to Clean (Kaggle)

En este notebook limpio y analizo el dataset de observaciones de animales en Europa Central y Oriental. El archivo tiene varios problemas de calidad (nulos, duplicados, texto mal escrito, etc), así que primero lo reviso y después lo voy arreglando paso a paso.

In [ ]:
import pandas as pd
import numpy as np

Subo el archivo csv (versión dirty del dataset) directo desde mi computador.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("animal_data_dirty1-selected-columns.csv", sep=";")
df.head()

## 1. Diagnóstico inicial

In [ ]:
# tamaño del dataframe (filas, columnas)
df.shape

In [ ]:
# columnas y tipo de dato de cada una
df.dtypes

In [ ]:
# valores faltantes por columna
df.isnull().sum()

In [ ]:
# registros duplicados
df.duplicated().sum()

In [ ]:
df["Animal type"].unique()

In [ ]:
df["Country"].unique()

In [ ]:
df["Gender"].value_counts()

**Problemas de calidad que encontré:**

1. La columna `Animal type` tiene varias formas distintas para el mismo animal, por ejemplo `European bison`, `European bison™`, `European bisson` y `European buster` en realidad son todos bisontes, pero al estar escritos distinto Python los cuenta como categorías diferentes. Lo mismo pasa con `lynx` y `lynx?`, o `hedgehog`, `wedgehod` y `ledgehod`.
2. La columna `Country` mezcla nombres completos con abreviaciones (`Poland` y `PL`, `Hungary` y `HU`) y además tiene errores de tipeo como `Hungry` en vez de `Hungary`, y `Australia` en vez de `Austria` (que sí tiene sentido geográfico para este dataset, Australia no).
3. Hay 166 filas duplicadas exactamente iguales.
4. `Weight kg` y `Body Length cm` tienen valores negativos en algunos registros, lo cual no puede ser (un peso o una longitud no pueden ser negativos), probablemente un error al momento de digitar el dato.
5. La columna `Animal code` está vacía en el 100% de las filas y `Animal name` casi vacía, así que no aportan mucho al análisis.

## 2. Limpieza de datos

In [ ]:
# Animal code no tiene ni un solo dato y Animal name casi no tiene datos, las elimino
df = df.drop(columns=["Animal code", "Animal name"])

In [ ]:
# hay filas donde Animal type está vacío, sin saber qué animal es no puedo usarlas para el análisis
df = df.dropna(subset=["Animal type"])
df.shape

In [ ]:
# el peso y el largo no pueden ser negativos, seguro fue un error de digitación, uso valor absoluto
df["Weight kg"] = df["Weight kg"].abs()
df["Body Length cm"] = df["Body Length cm"].abs()

In [ ]:
# relleno los numéricos faltantes con la mediana (menos sensible a valores extremos que el promedio)
df["Weight kg"] = df["Weight kg"].fillna(df["Weight kg"].median())
df["Body Length cm"] = df["Body Length cm"].fillna(df["Body Length cm"].median())
df["Latitude"] = df["Latitude"].fillna(df["Latitude"].median())
df["Longitude"] = df["Longitude"].fillna(df["Longitude"].median())

In [ ]:
# para el género, en vez de inventar un valor uso "not determined" ya que esa categoría ya existe en el dataset
df["Gender"] = df["Gender"].fillna("not determined")

In [ ]:
# reviso que ya no queden nulos
df.isnull().sum()

In [ ]:
# quito espacios raros al inicio/final del texto
df["Animal type"] = df["Animal type"].str.strip()

# diccionario para unificar los nombres de animales que estaban mal escritos
animal_type_map = {
    "European bison™": "European bison",
    "European bisson": "European bison",
    "European buster": "European bison",
    "lynx": "Lynx",
    "lynx?": "Lynx",
    "red squirel": "Red squirrel",
    "red squirrel": "Red squirrel",
    "red squirrell": "Red squirrel",
    "hedgehog": "Hedgehog",
    "wedgehod": "Hedgehog",
    "ledgehod": "Hedgehog",
}

df["Animal type"] = df["Animal type"].replace(animal_type_map)
df["Animal type"].value_counts()

In [ ]:
# lo mismo para el país: unifico abreviaciones y corrijo errores de tipeo
country_map = {
    "PL": "Poland",
    "DE": "Germany",
    "HU": "Hungary",
    "Hungry": "Hungary",
    "CZ": "Czech Republic",
    "CC": "Czech Republic",
    "Czech": "Czech Republic",
    "Australia": "Austria",
}

df["Country"] = df["Country"].replace(country_map)
df["Country"] = df["Country"].fillna(df["Country"].mode()[0])
df["Country"].value_counts()

In [ ]:
# reviso duplicados otra vez, ahora que ya limpié las categorías pueden haber aparecido más
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

In [ ]:
# la fecha está como texto (object), la paso a formato fecha
df["Observation date"] = pd.to_datetime(df["Observation date"], format="mixed", dayfirst=True)
df.dtypes

In [ ]:
# verifico que todo haya quedado bien: sin nulos, sin duplicados, categorías limpias
print("Nulos:\n", df.isnull().sum())
print("\nDuplicados:", df.duplicated().sum())
print("\nAnimales:", df["Animal type"].unique())
print("\nPaíses:", df["Country"].unique())
print("\nForma final del dataframe:", df.shape)

## 3. NumPy

In [ ]:
# creo una categoría de peso: si el animal pesa más que la mediana del dataset lo marco como "Grande", si no como "Pequeño"
df["Weight category"] = np.where(df["Weight kg"] > df["Weight kg"].median(), "Grande", "Pequeño")
df["Weight category"].value_counts()

In [ ]:
mean_weight = np.mean(df["Weight kg"])
median_weight = np.median(df["Weight kg"])
std_weight = np.std(df["Weight kg"])

print("Promedio:", round(mean_weight, 2))
print("Mediana:", round(median_weight, 2))
print("Desviación estándar:", round(std_weight, 2))

El promedio de peso (≈47 kg) está muy por encima de la mediana (≈0.35 kg), y la desviación estándar es enorme (≈170). Esto pasa porque el dataset mezcla animales muy chicos (ardillas, erizos) con uno gigante como el bisonte europeo, entonces el promedio se ve "inflado" por esos pocos valores altos. La mediana representa mejor el "animal típico" del dataset.

## 4. Análisis con groupby()

In [ ]:
# cantidad de observaciones por tipo de animal
df.groupby("Animal type").size()

In [ ]:
# peso promedio según el tipo de animal
df.groupby("Animal type")["Weight kg"].mean().round(2)

In [ ]:
# largo del cuerpo promedio según el tipo de animal
df.groupby("Animal type")["Body Length cm"].mean().round(2)

In [ ]:
# peso promedio según el país
df.groupby("Country")["Weight kg"].mean().round(2)

In [ ]:
# extra: cantidad de observaciones por país
df.groupby("Country").size()

## 5. Interpretación de resultados

1. **Red squirrel es el animal más observado**, con 413 registros, seguido de Hedgehog con 278. Esto indica que el equipo de investigación se enfocó bastante más en especies pequeñas y probablemente más fáciles de avistar que en especies grandes como el bisonte (solo 63 registros).

2. **El peso promedio cambia muchísimo según la especie**: European bison pesa en promedio 588 kg mientras que Red squirrel pesa apenas 0.30 kg. Esto confirma que el promedio general de peso del dataset completo (≈47 kg) no sirve para describir "al animal típico", porque en realidad está mezclando especies de tamaños totalmente distintos.

3. **Polonia (Poland) es el país con más observaciones**, 192 en total. Esto tiene sentido porque justamente ahí es donde se concentran los registros de bisonte europeo, uno de los animales más buscados por el equipo.

4. **El peso promedio por país también refleja qué animales se observaron en cada lugar**: Polonia tiene el promedio más alto (130 kg aprox) por los bisontes, mientras que Alemania tiene el promedio más bajo (0.46 kg aprox), porque ahí solo se registraron animales pequeños como ardillas o erizos.

5. **La desviación estándar del peso es altísima (≈170) comparada con la mediana (≈0.35)**, lo que confirma otra vez que hay mucha dispersión en los datos por juntar animales de tamaños tan diferentes en un mismo análisis.